# Phase-0 v2 — Fix the 3 RNN classifiers

Problem in v1: BiGRU/BiRNN+Attn/BiRNN+Skip underperformed the plain ARX linear model on the same feature matrix. That's because the technical indicators (RSI, MACD, mom_5/10/20) already summarize history — stacking 20 copies of summarized history is not sequence learning.

Fixes in v2:
1. **Wavelet leak removed** — wavelet denoising applied globally to the full series was a look-ahead leak. Using raw `Close`.
2. **Sequence branch = raw daily returns for 60 days** (shape B×60×1). RNN actually gets temporal data to model.
3. **Static branch = today's indicators + FRED deltas** through an MLP. This is what the linear ARX sees.
4. **Hybrid fusion head** — concat sequence-pool + static-MLP → classifier.
5. **Training hygiene** — class-weighted BCE, gradient clipping, orthogonal GRU init, cosine LR with warmup, patience 12.

Path matches v1: `/content/drive/MyDrive/Quants /...`

In [ ]:
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from scipy.stats import beta as _beta
import warnings; warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42); np.random.seed(42)
print('Device:', DEVICE)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/Quants ')
WHEAT_18 = BASE / 'Investing.com' / 'US Wheat Futures Historical Data_2018.csv'
WHEAT_25 = BASE / 'Investing.com' / 'US Wheat Futures Historical Data_2025.csv'
FRED = BASE / 'FredMD_Dataset' / '2025-10-MD.csv'

## 1. Data — OHLCV + technical indicators (no wavelet)

In [ ]:
def parse_volume(v):
    if pd.isna(v): return np.nan
    v = str(v).replace(',', '').strip()
    if v in ('', '-'): return np.nan
    mult = 1.0
    if v.endswith('K'): mult, v = 1e3, v[:-1]
    elif v.endswith('M'): mult, v = 1e6, v[:-1]
    try: return float(v) * mult
    except: return np.nan

def load_ohlcv(p):
    df = pd.read_csv(p)
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').set_index('Date')
    for c in ['Price','Open','High','Low']:
        df[c] = df[c].astype(str).str.replace(',', '', regex=False).astype(float)
    df['Volume'] = df['Vol.'].apply(parse_volume)
    return df[['Price','Open','High','Low','Volume']].rename(columns={'Price':'Close'})

d1 = load_ohlcv(WHEAT_18); d2 = load_ohlcv(WHEAT_25)
ohlcv = pd.concat([d1, d2]).pipe(lambda d: d[~d.index.duplicated(keep='last')]).sort_index()
ohlcv['Volume'] = ohlcv['Volume'].ffill().bfill().fillna(ohlcv['Volume'].median())
print('OHLCV shape:', ohlcv.shape)

In [ ]:
def rsi(s, n=14):
    d = s.diff(); up = d.clip(lower=0); dn = -d.clip(upper=0)
    ru = up.ewm(alpha=1/n, adjust=False).mean(); rd = dn.ewm(alpha=1/n, adjust=False).mean()
    return 100 - 100/(1+ru/(rd+1e-9))

def macd(s, f=12, sl=26, sig=9):
    ef = s.ewm(span=f, adjust=False).mean(); es = s.ewm(span=sl, adjust=False).mean()
    m = ef - es; sg = m.ewm(span=sig, adjust=False).mean()
    return m, sg, m - sg

def bb_percent(s, n=20, k=2):
    ma = s.rolling(n).mean(); sd = s.rolling(n).std()
    return (s - (ma - k*sd)) / (2*k*sd + 1e-9)

def atr(df, n=14):
    tr = pd.concat([df['High']-df['Low'],
                    (df['High']-df['Close'].shift()).abs(),
                    (df['Low']-df['Close'].shift()).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1/n, adjust=False).mean()

c = ohlcv['Close']
stat = pd.DataFrame(index=ohlcv.index)
stat['mom_5']  = c.pct_change(5)
stat['mom_10'] = c.pct_change(10)
stat['mom_20'] = c.pct_change(20)
stat['rsi_14'] = rsi(c)
m, sg, h = macd(c); stat['macd']=m; stat['macd_sig']=sg; stat['macd_hist']=h
stat['bb_pct'] = bb_percent(c)
stat['atr_14'] = atr(ohlcv)
stat['atr_pct'] = stat['atr_14']/c
stat['hl_range'] = (ohlcv['High']-ohlcv['Low'])/c
stat['oc_ret'] = (ohlcv['Close']-ohlcv['Open'])/ohlcv['Open']
stat['vol_z'] = (ohlcv['Volume']-ohlcv['Volume'].rolling(60).mean())/(ohlcv['Volume'].rolling(60).std()+1e-9)
stat['vol_chg'] = ohlcv['Volume'].pct_change().clip(-5,5)
print('Static features:', stat.shape, list(stat.columns))

In [ ]:
# FRED monthly deltas with 1-month publication lag
fred_raw = pd.read_csv(FRED, skiprows=[1])
fred_raw['sasdate'] = pd.to_datetime(fred_raw['sasdate'], format='%m/%d/%Y')
fred_raw = fred_raw.set_index('sasdate').sort_index()
keep = ['RPI','W875RX1','CMRMTSPLx','IPFPNSS','FEDFUNDS','TB3MS','GS10','BAA','AAA',
        'T10YFFM','T5YFFM','T1YFFM','BAAFFM','S&P 500','EXCAUSx','EXUSUKx','CPIAUCSL',
        'PPICMM','OILPRICEx','UMCSENTx']
keep = [k for k in keep if k in fred_raw.columns]
fred = fred_raw[keep].pct_change().replace([np.inf,-np.inf], np.nan)
fred.columns = [f'fred_{c}_d' for c in fred.columns]
fred.index = fred.index + pd.DateOffset(months=1)
fred_daily = fred.reindex(pd.date_range(fred.index.min(), ohlcv.index.max(), freq='D')).ffill()

# Static features combined
X_static = stat.join(fred_daily, how='left').replace([np.inf,-np.inf], np.nan).ffill()

# Sequence feature = raw daily return (bounded, clean)
seq_ret = ohlcv['Close'].pct_change().clip(-0.2, 0.2).to_frame('ret1')
seq_vol = ohlcv['Volume'].pct_change().clip(-5, 5).to_frame('volchg')
X_seq_raw = seq_ret.join(seq_vol).replace([np.inf,-np.inf], np.nan).ffill().fillna(0.0)
print('X_static:', X_static.shape, '  X_seq_raw:', X_seq_raw.shape)

## 2. Target and splits

In [ ]:
H = 5
y_h = (ohlcv['Close'].pct_change(H).shift(-H) > 0).astype(int)

aligned = X_static.join(y_h.rename('y'), how='inner').dropna()
aligned = aligned.join(X_seq_raw, how='left').dropna()
print('Aligned:', aligned.shape)

STAT_COLS = [c for c in X_static.columns]
SEQ_COLS = ['ret1', 'volchg']

n = len(aligned); i1 = int(n*0.70); i2 = int(n*0.85)
train = aligned.iloc[:i1]; val = aligned.iloc[i1:i2]; test = aligned.iloc[i2:]
print(f'train  n={len(train)}  {train.index.min().date()}..{train.index.max().date()}')
print(f'val    n={len(val)}   {val.index.min().date()}..{val.index.max().date()}')
print(f'test   n={len(test)}  {test.index.min().date()}..{test.index.max().date()}')

stat_scaler = StandardScaler().fit(train[STAT_COLS])
# Sequence is raw returns — mean~0; we z-score using train stats only
seq_mu  = train[SEQ_COLS].mean().values
seq_std = train[SEQ_COLS].std().values + 1e-9

LOOKBACK = 60
def build_sets(split):
    S = stat_scaler.transform(split[STAT_COLS]).astype(np.float32)
    Q = ((split[SEQ_COLS].values - seq_mu) / seq_std).astype(np.float32)
    y = split['y'].values.astype(np.float32)
    seq, stat_last, yy = [], [], []
    for i in range(LOOKBACK, len(split)):
        seq.append(Q[i-LOOKBACK:i])
        stat_last.append(S[i])
        yy.append(y[i])
    return np.array(seq), np.array(stat_last), np.array(yy)

Xs_tr, Xf_tr, y_tr = build_sets(train)
Xs_va, Xf_va, y_va = build_sets(val)
Xs_te, Xf_te, y_te = build_sets(test)
print('seq:', Xs_tr.shape, '  static:', Xf_tr.shape)
print('class balance train/val/test:', y_tr.mean().round(3), y_va.mean().round(3), y_te.mean().round(3))

## 3. Baseline — LogReg on static only (what we're trying to beat)

In [ ]:
base = LogisticRegression(max_iter=3000, C=0.5).fit(Xf_tr, y_tr)
p_te = base.predict_proba(Xf_te)[:,1]
base_res = {'test_acc': accuracy_score(y_te, p_te>0.5),
            'test_auc': roc_auc_score(y_te, p_te),
            'test_f1':  f1_score(y_te, p_te>0.5)}
print('LogReg (static only):', base_res)

## 4. Shared hybrid training loop

In [ ]:
def orth_init_rnn(rnn):
    for name, p in rnn.named_parameters():
        if 'weight_hh' in name: nn.init.orthogonal_(p)
        elif 'weight_ih' in name: nn.init.kaiming_normal_(p)
        elif 'bias' in name: nn.init.zeros_(p)

def train_hybrid(model, epochs=80, bs=64, lr=1e-3, wd=1e-4, patience=12, clip=1.0):
    model = model.to(DEVICE)
    pos_weight = torch.tensor([(1 - y_tr.mean()) / max(y_tr.mean(), 1e-6)], device=DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=10, T_mult=2)
    ds = TensorDataset(torch.from_numpy(Xs_tr), torch.from_numpy(Xf_tr), torch.from_numpy(y_tr))
    ld = DataLoader(ds, batch_size=bs, shuffle=True)
    Xs_va_t = torch.from_numpy(Xs_va).to(DEVICE); Xf_va_t = torch.from_numpy(Xf_va).to(DEVICE)
    Xs_te_t = torch.from_numpy(Xs_te).to(DEVICE); Xf_te_t = torch.from_numpy(Xf_te).to(DEVICE)
    best_va = -1; best_state = None; bad = 0
    for ep in range(epochs):
        model.train()
        for xs, xf, yy in ld:
            xs, xf, yy = xs.to(DEVICE), xf.to(DEVICE), yy.to(DEVICE)
            opt.zero_grad()
            logit = model(xs, xf).squeeze(-1)
            loss = loss_fn(logit, yy)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            p_va = torch.sigmoid(model(Xs_va_t, Xf_va_t).squeeze(-1)).cpu().numpy()
        va_auc = roc_auc_score(y_va, p_va)
        if va_auc > best_va:
            best_va, best_state, bad = va_auc, {k:v.clone() for k,v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        p_tr = torch.sigmoid(model(torch.from_numpy(Xs_tr).to(DEVICE), torch.from_numpy(Xf_tr).to(DEVICE)).squeeze(-1)).cpu().numpy()
        p_te = torch.sigmoid(model(Xs_te_t, Xf_te_t).squeeze(-1)).cpu().numpy()
    return {'epochs_used': ep+1, 'val_auc': best_va,
            'train_acc': accuracy_score(y_tr, p_tr>0.5),
            'test_acc':  accuracy_score(y_te, p_te>0.5),
            'test_auc':  roc_auc_score(y_te, p_te),
            'test_f1':   f1_score(y_te, p_te>0.5)}, p_te

## 5. Model 1 — Hybrid BiGRU (last hidden + static MLP)

In [ ]:
class HybridBiGRU(nn.Module):
    def __init__(self, n_seq, n_stat, h_seq=48, h_stat=32, drop=0.3):
        super().__init__()
        self.rnn = nn.GRU(n_seq, h_seq, num_layers=2, batch_first=True,
                          bidirectional=True, dropout=drop)
        orth_init_rnn(self.rnn)
        self.mlp = nn.Sequential(nn.Linear(n_stat, h_stat), nn.ReLU(),
                                 nn.Dropout(drop), nn.Linear(h_stat, h_stat))
        self.head = nn.Sequential(nn.LayerNorm(2*h_seq + h_stat), nn.Dropout(drop),
                                  nn.Linear(2*h_seq + h_stat, 1))
    def forward(self, xs, xf):
        out, _ = self.rnn(xs)
        s = out[:, -1, :]                 # last hidden state
        f = self.mlp(xf)
        return self.head(torch.cat([s, f], dim=-1))

res_bigru, p_bigru = train_hybrid(HybridBiGRU(Xs_tr.shape[-1], Xf_tr.shape[-1]))
print('HybridBiGRU:', res_bigru)

## 6. Model 2 — Hybrid BiRNN + Attention

In [ ]:
class HybridBiRNNAttn(nn.Module):
    def __init__(self, n_seq, n_stat, h_seq=48, h_stat=32, drop=0.3):
        super().__init__()
        self.rnn = nn.GRU(n_seq, h_seq, num_layers=2, batch_first=True,
                          bidirectional=True, dropout=drop)
        orth_init_rnn(self.rnn)
        self.attn = nn.Linear(2*h_seq, 1)
        self.mlp = nn.Sequential(nn.Linear(n_stat, h_stat), nn.ReLU(),
                                 nn.Dropout(drop), nn.Linear(h_stat, h_stat))
        self.head = nn.Sequential(nn.LayerNorm(2*h_seq + h_stat), nn.Dropout(drop),
                                  nn.Linear(2*h_seq + h_stat, 1))
    def forward(self, xs, xf):
        out, _ = self.rnn(xs)
        a = torch.softmax(self.attn(out), dim=1)
        ctx = (a * out).sum(dim=1)
        f = self.mlp(xf)
        return self.head(torch.cat([ctx, f], dim=-1))

res_attn, p_attn = train_hybrid(HybridBiRNNAttn(Xs_tr.shape[-1], Xf_tr.shape[-1]))
print('HybridBiRNN+Attn:', res_attn)

## 7. Model 3 — Hybrid BiRNN + Skip

In [ ]:
class HybridBiRNNSkip(nn.Module):
    def __init__(self, n_seq, n_stat, h_seq=48, h_stat=32, drop=0.3):
        super().__init__()
        self.rnn1 = nn.GRU(n_seq, h_seq, batch_first=True, bidirectional=True)
        self.rnn2 = nn.GRU(2*h_seq, h_seq, batch_first=True, bidirectional=True)
        orth_init_rnn(self.rnn1); orth_init_rnn(self.rnn2)
        self.skip_proj = nn.Linear(n_seq, 2*h_seq)
        self.drop = nn.Dropout(drop)
        self.mlp = nn.Sequential(nn.Linear(n_stat, h_stat), nn.ReLU(),
                                 nn.Dropout(drop), nn.Linear(h_stat, h_stat))
        self.head = nn.Sequential(nn.LayerNorm(2*h_seq + h_stat), nn.Dropout(drop),
                                  nn.Linear(2*h_seq + h_stat, 1))
    def forward(self, xs, xf):
        r1, _ = self.rnn1(xs)
        r1 = self.drop(r1 + self.skip_proj(xs))
        r2, _ = self.rnn2(r1)
        r2 = r2 + r1
        s = r2[:, -1, :]
        f = self.mlp(xf)
        return self.head(torch.cat([s, f], dim=-1))

res_skip, p_skip = train_hybrid(HybridBiRNNSkip(Xs_tr.shape[-1], Xf_tr.shape[-1]))
print('HybridBiRNN+Skip:', res_skip)

## 8. Summary + Clopper–Pearson CI on accuracy

Checks whether test accuracy is *statistically* above 50% (not just nominally).

In [ ]:
def cp_ci(acc, n, alpha=0.05):
    k = int(round(acc*n))
    lo = _beta.ppf(alpha/2, k, n-k+1) if k>0 else 0.0
    hi = _beta.ppf(1-alpha/2, k+1, n-k) if k<n else 1.0
    return lo, hi

n_te = len(y_te)
rows = []
for name, r in [('LogReg (static only)',  {'test_acc':base_res['test_acc'],'test_auc':base_res['test_auc'],'test_f1':base_res['test_f1'],'train_acc':None}),
                ('HybridBiGRU',            res_bigru),
                ('HybridBiRNN+Attn',       res_attn),
                ('HybridBiRNN+Skip',       res_skip)]:
    lo, hi = cp_ci(r['test_acc'], n_te)
    rows.append((name, r.get('train_acc'), r['test_acc'], f'[{lo:.3f},{hi:.3f}]',
                 r['test_auc'], r['test_f1']))
summary = pd.DataFrame(rows, columns=['model','train_acc','test_acc','95% CI','test_auc','test_f1'])
print(summary.to_string(index=False))
summary.to_csv('/content/drive/MyDrive/Quants /phase0_v2_summary.csv', index=False)